# TabDPT Classifier — DIMER end-to-end tutorial

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/tabdpt-classifier-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/tabdpt-classifier-pipeline/blob/main/tutorials/tabdpt_classifier_colab.ipynb)
[![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Layer6%2FTabDPT-ffcc4d?style=flat)](https://huggingface.co/Layer6/TabDPT)
[![Upstream](https://img.shields.io/badge/Upstream-layer6ai--labs%2FTabDPT--inference-181717?style=flat&logo=github&logoColor=white)](https://github.com/layer6ai-labs/TabDPT-inference)
[![arXiv](https://img.shields.io/badge/arXiv-2608.01400-b31b1b.svg)](https://arxiv.org/abs/2608.01400)

**Profile:** `E2E`  
**Notebook specification:** DIMER Notebook Specification `1.0`  
**Repository code revision exercised:** `d9a99b464b14633b6561662b8b59c402512474ea`

This notebook demonstrates DIMER tabular **classification** through the repository's production-facing `TabDPTClassificationPipeline`. TabDPT is an in-context learner: `fit()` fits/records preprocessing state and registers a labelled support table for inference. **It does not perform gradient training or fine-tune the TabDPT weights.**

The upstream TabDPT v1.2 package and immutable checkpoint supply the foundation model. This repository adds DIMER-facing model provenance and checksum verification, mixed-table preprocessing, schema enforcement, deterministic controls, evaluation helpers, strict artifact validation, and the `tabdpt-dimer-context-v3` serving-artifact contract.

**By the end of this notebook you will be able to:**
- install an immutable repository revision with a pinned tutorial environment;
- resolve and SHA-256 verify the exact TabDPT v1.2 model weights;
- load a public sample or gated bring-your-own CSV data and validate its schema;
- condition TabDPT on labelled support data without gradient updates;
- evaluate classification with accuracy, log loss, ROC-AUC where defined, and a majority-class baseline;
- score genuinely separate new records and export machine-readable predictions;
- export the actual DIMER v3 serving artifact, validate/reload it from serialized files, and verify prediction equivalence.

**This notebook does not demonstrate:** gradient fine-tuning, regression, causal inference, calibrated deployment thresholds, or production fitness. Tutorial metrics are demonstration evidence only. Public tabular data may overlap directly or indirectly with upstream pretraining, so these metrics are not clean benchmark evidence.

Repository references: [README](https://github.com/kurtvalcorza/tabdpt-classifier-pipeline/blob/main/README.md), [model card](https://github.com/kurtvalcorza/tabdpt-classifier-pipeline/blob/main/MODEL_CARD.md), [dataset specification](https://github.com/kurtvalcorza/tabdpt-classifier-pipeline/blob/main/TABULAR_CLASSIFICATION_DATASET_SPEC.md), and [DIMER contract](https://github.com/kurtvalcorza/tabdpt-classifier-pipeline/blob/main/DIMER_CONTRACT.md).


## Prerequisites and runtime contract

- **Environment:** fresh Google Colab or compatible Jupyter runtime; Python 3.11–3.13.
- **Accelerator:** GPU recommended. CPU is supported but may be slower. The demonstrated path forces `use_flash=False`, which is portable to Tesla T4 (`sm_75`) and newer GPUs.
- **Network:** required once to clone the pinned repository revision and acquire the pinned Hugging Face checkpoint unless already cached.
- **Data:** the default sample requires no private data. BYOD modes are optional and gated.
- **Privacy:** uploaded files stay in the notebook runtime unless explicitly exported elsewhere. Do not upload confidential, restricted, or sensitive data to a hosted notebook environment unless authorized.
- **DIMER service controls:** default fitted-support ceiling `10,000` rows; default per-ensemble context `2,048`; supported DIMER context range `128–16,384`. These are service controls, not intrinsic upstream model limits.

**Execution assumptions:** `compile_model=False`, `use_flash=False`, no notebook-level quantization, and no explicit mixed-precision override. Device placement and numerical precision otherwise follow the pinned TabDPT/PyTorch implementation. The tutorial seeds Python/NumPy/PyTorch through the pipeline and explicitly seeds data splitting and ensemble inference. These controls improve reproducibility but **do not guarantee bitwise-identical results across devices, CUDA/library builds, or hardware kernels**; backend/device differences can remain sources of run-to-run variation.

The setup cell refuses a non-fresh Python process where PyTorch is already imported because the pinned install may replace core packages and would otherwise create a hidden restart boundary.


In [ ]:
import sys
if "torch" in sys.modules:
    raise RuntimeError(
        "Start from a fresh runtime: PyTorch is already imported, and the pinned tutorial install "
        "must complete before core ML packages are loaded."
    )

REPO_REVISION = "d9a99b464b14633b6561662b8b59c402512474ea"
REPO_DIR = "/content/tabdpt-classifier-pipeline"

!rm -rf "$REPO_DIR"
!git clone -q https://github.com/kurtvalcorza/tabdpt-classifier-pipeline.git "$REPO_DIR"
!git -C "$REPO_DIR" checkout -q "$REPO_REVISION"
!python -m pip install -q -r "$REPO_DIR/tutorials/requirements-colab.txt"
!python -m pip install -q --no-deps "$REPO_DIR"


## 1. Verify the runtime and model provenance

The next cell prints effective runtime identity, then resolves the exact model checkpoint through repository code. The resolver is pinned to `Layer6/TabDPT` at an immutable Hugging Face revision and verifies SHA-256 before model construction. Model-repository remote Python code is not used: the checkpoint is `.safetensors`, while inference code comes from the pinned `tabdpt==1.2.0` package.

A successful cell establishes **identity and byte integrity**, not model quality or deployment safety.


In [ ]:
import importlib.metadata as mdlib
import platform
import sys
import torch

from tabdpt_classifier_pipeline import (
    TABDPT_HF_REPO,
    TABDPT_HF_REVISION,
    TABDPT_UPSTREAM_CODE_COMMIT,
    TABDPT_WEIGHT_FILENAME,
    TABDPT_WEIGHT_SHA256,
    TabDPTClassificationPipeline,
    resolve_tabdpt_weights,
    validate_dimer_artifact,
)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
for package in ["tabdpt", "torch", "numpy", "pandas", "scikit-learn", "huggingface-hub", "pyarrow"]:
    print(f"{package}:", mdlib.version(package))
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA runtime:", torch.version.cuda)
print("Tutorial settings: compile_model=False, use_flash=False, no notebook-level quantization")

weights = resolve_tabdpt_weights()
print("Model repository:", TABDPT_HF_REPO)
print("Model revision:", TABDPT_HF_REVISION)
print("Upstream code commit:", TABDPT_UPSTREAM_CODE_COMMIT)
print("Weight file:", TABDPT_WEIGHT_FILENAME)
print("Expected/verified SHA-256:", TABDPT_WEIGHT_SHA256)
print("Resolved local path:", weights)


## 2. Load the default sample or bring your own data

Choose one mode below.

- `sample` — public `sklearn.datasets.load_breast_cancer` data with deterministic support/evaluation/new-record partitions.
- `upload_single` — upload one CSV containing features plus the target; deterministic stratified support/evaluation/new-record partitions are created. Random splitting assumes rows are sufficiently independent.
- `upload_presplit` — upload `train.csv`, `val.csv`, and `new.csv`; existing partitions are preserved. `train.csv` and `val.csv` require the target; `new.csv` must be unlabelled.

**Expected schema before upload:** one classification target column (default `target`) plus one or more uniquely named feature columns. The target must have at least two classes in support data; every evaluation class must occur in support data. `new.csv` must match the fitted feature schema, except configured drop columns may also be present.

For temporal, grouped, panel, patient, device, household, spatial, repeated-entity, or otherwise leakage-sensitive data, use `upload_presplit` with domain-valid partitions. Do not use random row splitting merely because it is convenient.


In [ ]:
DATA_MODE = "sample"  # @param ["sample", "upload_single", "upload_presplit"]
TARGET_COLUMN = "target"  # @param {type:"string"}
SEED = 42  # @param {type:"integer"}

import csv
from collections import Counter
from pathlib import Path
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

DATA_DIR = Path("/content/tabdpt-tutorial-data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

def read_csv_strict(path: Path) -> pd.DataFrame:
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        reader = csv.reader(handle)
        try:
            header = next(reader)
        except StopIteration as exc:
            raise ValueError(f"{path.name} is empty") from exc
    duplicates = sorted(name for name, count in Counter(header).items() if count > 1)
    if duplicates:
        raise ValueError(f"{path.name} contains duplicate column names: {duplicates}")
    return pd.read_csv(path)

def validate_labeled(frame: pd.DataFrame, name: str) -> None:
    if TARGET_COLUMN not in frame.columns:
        raise ValueError(f"{name} is missing target column {TARGET_COLUMN!r}")
    if frame.columns.duplicated().any():
        raise ValueError(f"{name} contains duplicate column names")
    if frame[TARGET_COLUMN].isna().any():
        raise ValueError(f"{name} target contains missing values")
    if frame.drop(columns=[TARGET_COLUMN]).shape[1] == 0:
        raise ValueError(f"{name} must contain at least one feature column")

if DATA_MODE == "sample":
    frame = load_breast_cancer(as_frame=True).frame.copy()
    support_eval, new_labeled = train_test_split(
        frame, test_size=0.10, random_state=SEED, stratify=frame[TARGET_COLUMN]
    )
    support, evaluation = train_test_split(
        support_eval, test_size=2/9, random_state=SEED, stratify=support_eval[TARGET_COLUMN]
    )
    new_records = new_labeled.drop(columns=[TARGET_COLUMN]).copy()
    DATA_PROVENANCE = "scikit-learn breast cancer dataset; public tutorial sample"
elif DATA_MODE in {"upload_single", "upload_presplit"}:
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError(
            "Upload modes require Google Colab. In local Jupyter, place the expected CSV files "
            f"under {DATA_DIR} and adapt this acquisition cell only."
        ) from exc
    uploaded = files.upload()
    for name, payload in uploaded.items():
        (DATA_DIR / Path(name).name).write_bytes(payload)

    if DATA_MODE == "upload_single":
        csv_files = sorted(DATA_DIR.glob("*.csv"))
        if len(csv_files) != 1:
            raise ValueError("upload_single requires exactly one CSV file")
        frame = read_csv_strict(csv_files[0])
        validate_labeled(frame, csv_files[0].name)
        support_eval, new_labeled = train_test_split(
            frame, test_size=0.10, random_state=SEED, stratify=frame[TARGET_COLUMN]
        )
        support, evaluation = train_test_split(
            support_eval, test_size=2/9, random_state=SEED, stratify=support_eval[TARGET_COLUMN]
        )
        new_records = new_labeled.drop(columns=[TARGET_COLUMN]).copy()
        DATA_PROVENANCE = f"user upload: {csv_files[0].name}; deterministic stratified 70/20/10 split"
    else:
        expected = {name: DATA_DIR / name for name in ("train.csv", "val.csv", "new.csv")}
        missing = [name for name, path in expected.items() if not path.exists()]
        if missing:
            raise ValueError(f"upload_presplit is missing required files: {missing}")
        support = read_csv_strict(expected["train.csv"])
        evaluation = read_csv_strict(expected["val.csv"])
        new_records = read_csv_strict(expected["new.csv"])
        validate_labeled(support, "train.csv")
        validate_labeled(evaluation, "val.csv")
        if TARGET_COLUMN in new_records.columns:
            raise ValueError("new.csv must be unlabelled; remove the target column")
        DATA_PROVENANCE = "user-provided train.csv/val.csv/new.csv partitions preserved"
else:
    raise ValueError(f"Unsupported DATA_MODE: {DATA_MODE!r}")

validate_labeled(support, "support")
validate_labeled(evaluation, "evaluation")
support_classes = set(support[TARGET_COLUMN].map(str))
evaluation_classes = set(evaluation[TARGET_COLUMN].map(str))
missing_eval_classes = sorted(evaluation_classes - support_classes)
if missing_eval_classes:
    raise ValueError(f"Evaluation contains classes absent from support data: {missing_eval_classes}")

support_hashes = set(pd.util.hash_pandas_object(support, index=False).astype(str))
evaluation_hashes = set(pd.util.hash_pandas_object(evaluation, index=False).astype(str))
overlap = support_hashes & evaluation_hashes
if overlap:
    raise ValueError(f"Detected {len(overlap)} exact row value(s) duplicated across support and evaluation partitions")

if len(support) > 10_000:
    raise ValueError(
        f"Support has {len(support):,} rows, exceeding the DIMER default support ceiling of 10,000. "
        "This tutorial refuses silent capping; provide an explicit, documented support subset."
    )

print("Data provenance:", DATA_PROVENANCE)
print("Support shape:", support.shape)
print("Evaluation shape:", evaluation.shape)
print("New-record shape:", new_records.shape)
print("Support classes:", sorted(support_classes))
print("Evaluation classes:", sorted(evaluation_classes))
print("Exact support/evaluation row-overlap check: PASS")


## 3. Condition the model on support data

`TabDPTClassificationPipeline.fit()` is the repository's supported conditioning API. It fits the repository's mixed-table preprocessing state and registers the labelled support context with the upstream estimator. **No TabDPT weight receives a gradient update.** Missing categorical values receive a dedicated code; categorical values not observed in support data receive a distinct unknown code at evaluation/inference.

Successful execution means that the support schema, labels, preprocessing, pinned model bytes, and in-context state were accepted. It does not establish generalization quality.


In [ ]:
pipe = TabDPTClassificationPipeline(
    model_weight_path=weights,
    compile_model=False,
    use_flash=False,
    seed=SEED,
)
pipe.fit(support, target_column=TARGET_COLUMN, seed=SEED)

def report_category_drift(frame, label):
    for column, mapping in pipe.feature_encoder.category_maps.items():
        if column not in frame.columns:
            continue
        observed = {str(value) for value in frame[column].dropna().tolist()}
        unseen = sorted(observed - set(mapping))
        if unseen:
            preview = unseen[:5]
            print(f"WARNING: {label}.{column} has {len(unseen)} unseen categorical value(s); examples={preview}. They use the fitted encoder's unknown code.")

report_category_drift(evaluation, "evaluation")
report_category_drift(new_records, "new_records")
print("Fitted feature count:", len(pipe.feature_encoder.feature_columns))
print("Class order:", pipe.class_labels_)
print("Adaptation semantics: preprocessing fit + in-context support conditioning; no gradient training")


## 4. Evaluate and compare a trivial baseline

The repository reports **accuracy** (discrete correctness), **log loss** (sensitivity to the score assigned to the true class), and **ROC-AUC** for binary classification when defined (ranking discrimination across thresholds). Important complementary error modes remain; no single metric establishes overall model quality.

These are **single-holdout tutorial metrics**. No dispersion estimate is computed, so the displayed values are not claimed stable across datasets or runs. `predict_proba()` exposes class scores with probability-like normalization, but this tutorial does **not** establish calibration. The default decision rule is **argmax**. Application-specific calibration and threshold selection require labelled deployment-like evidence.

A majority-class accuracy baseline is included. Ensemble/context subsampling is explicitly reported when it applies rather than silently reducing the effective support context.


In [ ]:
import json
from sklearn.metrics import accuracy_score

INFERENCE = {
    "n_ensembles": 2,
    "context_size": 512,
    "batch_size": 512,
    "temperature": 1.0,
    "seed": SEED,
}
if not (128 <= INFERENCE["context_size"] <= 16_384):
    raise ValueError("context_size is outside the DIMER supported range 128..16384")
if len(support) > INFERENCE["context_size"]:
    print(
        f"Context reduction active: {len(support)} support rows are retained in the artifact, "
        f"while each ensemble inference uses up to {INFERENCE['context_size']} support rows via "
        "the pipeline's balanced context-reduction policy. Selection is seeded, but backend/hardware "
        "differences can still prevent bitwise determinism."
    )
else:
    print(f"Context reduction inactive: support rows={len(support)} <= context_size={INFERENCE['context_size']}")

metrics = pipe.evaluate(evaluation, **INFERENCE)
majority_class = support[TARGET_COLUMN].map(str).mode().iloc[0]
baseline_accuracy = float(
    accuracy_score(evaluation[TARGET_COLUMN].map(str), [majority_class] * len(evaluation))
)
evaluation_report = {
    "estimationProcedure": "single deterministic holdout; no model selection uses this holdout",
    "tutorialEvidenceOnly": True,
    "modelMetrics": metrics,
    "majorityClassBaseline": {"class": majority_class, "accuracy": baseline_accuracy},
}
print(json.dumps(evaluation_report, indent=2))


## 5. Run inference on separate new records

These records are separate from the evaluation partition. The output preserves a stable `row_id`, a discrete `prediction`, and one score column per class in exact `pipe.class_labels_` order. Scores support argmax classification/ranking, but this notebook does not establish calibrated probabilities or a universal deployment threshold.


In [ ]:
import pandas as pd

new_scores = pipe.predict_proba(new_records, **INFERENCE)
new_predictions = pipe.predict(new_records, **INFERENCE)
prediction_table = pd.DataFrame({"row_id": new_records.index.astype(str), "prediction": new_predictions.astype(str)})
for class_name in pipe.class_labels_:
    prediction_table[f"score_{class_name}"] = new_scores[class_name].to_numpy()
print(prediction_table.head())
print("Score-column class order:", pipe.class_labels_)


## 6. Export machine-readable outputs, provenance, and the DIMER serving artifact

The deployable TabDPT serving state is **not the checkpoint alone**. It also requires the labelled support context and fitted preprocessing state. This notebook writes `predictions.csv`, `metrics.json`, `provenance.json`, plus the actual DIMER v3 artifact: `artifact.json` and `training_context.parquet`.

Because the artifact contains support data, apply the source dataset's confidentiality, licensing, retention, and disclosure controls. The manifest contains no credentials. Its support-table SHA-256 and size establish internal consistency; they do not authenticate the sender.


In [ ]:
import hashlib
import importlib.metadata as mdlib
import json
from pathlib import Path

OUTPUT_DIR = Path("/content/tabdpt-tutorial-output")
ARTIFACT_DIR = OUTPUT_DIR / "artifact"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

predictions_path = OUTPUT_DIR / "predictions.csv"
metrics_path = OUTPUT_DIR / "metrics.json"
provenance_path = OUTPUT_DIR / "provenance.json"
context_path = ARTIFACT_DIR / "training_context.parquet"
manifest_path = ARTIFACT_DIR / "artifact.json"

prediction_table.to_csv(predictions_path, index=False)
metrics_path.write_text(json.dumps(evaluation_report, indent=2) + "\n", encoding="utf-8")
support.to_parquet(context_path, index=False)
preprocessing_state = pipe.export_preprocessing_state()
manifest = {
    "format": "tabdpt-dimer-context-v3",
    "taskType": "tabular_classification",
    "targetColumn": TARGET_COLUMN,
    "dropColumns": list(preprocessing_state["dropColumns"]),
    "classNames": list(pipe.class_labels_),
    "runtimeConfig": {"fine_tune": False, **INFERENCE},
    "preprocessing": preprocessing_state,
    "baseModel": {
        "repo": TABDPT_HF_REPO,
        "revision": TABDPT_HF_REVISION,
        "filename": TABDPT_WEIGHT_FILENAME,
        "sha256": TABDPT_WEIGHT_SHA256,
        "upstreamCodeCommit": TABDPT_UPSTREAM_CODE_COMMIT,
    },
    "trainingContext": {
        "path": context_path.name,
        "sizeBytes": context_path.stat().st_size,
        "sha256": sha256_file(context_path),
    },
}
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8")

provenance = {
    "repository": "kurtvalcorza/tabdpt-classifier-pipeline",
    "repositoryRevision": REPO_REVISION,
    "notebookProfile": "E2E",
    "notebookSpecVersion": "1.0",
    "dataProvenance": DATA_PROVENANCE,
    "model": manifest["baseModel"],
    "adaptation": {
        "type": "preprocessing fit + in-context support conditioning; no gradient training",
        "targetColumn": TARGET_COLUMN,
        "supportRows": len(support),
        "supportSha256": manifest["trainingContext"]["sha256"],
        "seed": SEED,
    },
    "classOrder": list(pipe.class_labels_),
    "inference": INFERENCE,
    "runtime": {package: mdlib.version(package) for package in ["tabdpt", "torch", "numpy", "pandas", "scikit-learn", "huggingface-hub", "pyarrow"]},
    "useFlash": False,
}
provenance_path.write_text(json.dumps(provenance, indent=2, sort_keys=True) + "\n", encoding="utf-8")

print("Predictions:", predictions_path)
print("Metrics:", metrics_path)
print("Provenance:", provenance_path)
print("Artifact manifest:", manifest_path)
print("Artifact context:", context_path)


## 7. Verify the serialized artifact across a fresh reconstruction boundary

A successful in-memory object does not prove serialization works. The next cell records probe outputs, deletes the original pipeline object, copies **only** serialized artifact files into a fresh directory, calls the repository's strict pre-reconstruction validator, reconstructs through `load_artifact()`, and compares outputs.

The strict validator fails on missing/inconsistent format, task, model provenance, fitted preprocessing metadata, runtime controls, support path, file size (when declared), SHA-256, and unexpected unlisted files before model reconstruction. Discrete classes must match exactly; floating-point scores use `rtol=1e-6`, `atol=1e-7`.


In [ ]:
import gc
import numpy as np
import shutil
from pathlib import Path

probe = new_records.iloc[: min(10, len(new_records))].copy()
before_classes = pipe.predict(probe, **INFERENCE).astype(str).to_numpy()
before_scores = pipe.predict_proba(probe, **INFERENCE).to_numpy()
del pipe
gc.collect()

RELOAD_DIR = Path("/content/tabdpt-artifact-reload")
if RELOAD_DIR.exists():
    shutil.rmtree(RELOAD_DIR)
RELOAD_DIR.mkdir(parents=True)
shutil.copy2(manifest_path, RELOAD_DIR / "artifact.json")
shutil.copy2(context_path, RELOAD_DIR / "training_context.parquet")
validated_manifest, validated_context = validate_dimer_artifact(
    RELOAD_DIR / "artifact.json", strict_directory=True
)
print("Pre-reconstruction artifact validation: PASS", validated_context)

reloaded = TabDPTClassificationPipeline.load_artifact(
    RELOAD_DIR / "artifact.json",
    model_weight_path=weights,
    compile_model=False,
    use_flash=False,
    seed=SEED,
)
after_classes = reloaded.predict(probe, **INFERENCE).astype(str).to_numpy()
after_scores = reloaded.predict_proba(probe, **INFERENCE).to_numpy()
if not np.array_equal(before_classes, after_classes):
    raise AssertionError("Serialized artifact reload changed one or more discrete predictions")
if not np.allclose(before_scores, after_scores, rtol=1e-6, atol=1e-7):
    max_abs = float(np.max(np.abs(before_scores - after_scores)))
    raise AssertionError(f"Serialized artifact reload changed class scores; max abs diff={max_abs}")
print("Artifact reloaded from fresh directory: PASS")
print("Exact class-prediction equivalence: PASS")
print("Class-score equivalence within rtol=1e-6, atol=1e-7: PASS")


## Interpretation, limits, and next steps

A successful top-to-bottom run proves that this pinned tutorial environment can acquire and checksum-verify the specified TabDPT v1.2 weights, validate/condition on the demonstrated table, compute repository classification metrics, score separate new records, export machine-readable outputs and a DIMER v3 support-context artifact, strictly validate serialized artifact provenance/integrity, and reconstruct equivalent probe predictions from serialized files.

It **does not prove** that TabDPT is accurate, calibrated, fair, robust, secure, or suitable for a particular production domain. The public sample may overlap upstream pretraining. A single holdout has no uncertainty estimate. Random row splitting is inappropriate for leakage-sensitive data. Seeded execution is not a promise of bitwise equivalence across hardware/backends. Argmax is the default decision rule; application thresholds/calibration require domain-labelled evidence. The exported support context may contain sensitive source data and must be governed accordingly.

Before release, record a clean-runtime execution for the exact notebook/PR revision, including runtime, accelerator, outcome, and artifact-reload result. Static notebook validation is not a substitute for that execution evidence.

**Useful next experiments:** repeat evaluation across domain-valid splits; assess calibration on labelled deployment-like data; test schema drift/unseen categories; compare context and ensemble settings under a fixed protocol; and run the companion `ARTIFACT-INFERENCE` notebook using the exported artifact plus genuinely external new input.
